In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torchinfo

Understanding training pipeline with real world dataset

Getting data

In [35]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [36]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [37]:
df.describe()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,0.0
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,NaN
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,NaN
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,NaN
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,NaN
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,NaN
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,NaN
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,NaN


In [38]:
df.shape

(569, 33)

In [39]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [40]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Splitting data

In [41]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[: , 0], test_size= 0.2)

Scaling data

In [42]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [43]:
X_train

array([[ 0.88132053,  0.66818713,  0.90985304, ...,  0.5782597 ,
         0.25724321, -0.48610645],
       [ 0.24807238,  0.13752171,  0.32848497, ...,  1.48519151,
        -0.11055685, -0.06344881],
       [ 0.9608315 , -0.96533949,  0.93459211, ...,  0.46701764,
        -0.23368305, -0.24096502],
       ...,
       [ 0.04361558, -1.32988357,  0.00728942, ..., -0.10309793,
         0.2493505 , -1.2818299 ],
       [-1.55057953, -0.78537469, -1.5046799 , ..., -0.16026399,
         0.03466892,  1.02757145],
       [ 0.31054529,  0.73278988,  0.27158512, ...,  0.11166105,
         0.42772564,  0.80328113]])

In [44]:
y_train

262    M
94     M
29     M
25     M
66     B
      ..
522    B
293    B
225    B
114    B
184    M
Name: diagnosis, Length: 455, dtype: object

Label Encoder

In [45]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

NP arrays to Tensors

In [67]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

y_train_tensor

tensor([1., 1., 1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0.,
        1., 0., 0., 0., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 1., 0.,
        0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1.,
        0., 0., 0., 1., 0., 1., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1., 0., 1., 0.,
        0., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1.,
        0., 1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 1., 0., 0.,
        1., 1., 0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1.,
        0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0.,
        1., 1., 1., 0., 0., 0., 0., 1., 

In [68]:
X_train_tensor

tensor([[ 0.8813,  0.6682,  0.9099,  ...,  0.5783,  0.2572, -0.4861],
        [ 0.2481,  0.1375,  0.3285,  ...,  1.4852, -0.1106, -0.0634],
        [ 0.9608, -0.9653,  0.9346,  ...,  0.4670, -0.2337, -0.2410],
        ...,
        [ 0.0436, -1.3299,  0.0073,  ..., -0.1031,  0.2494, -1.2818],
        [-1.5506, -0.7854, -1.5047,  ..., -0.1603,  0.0347,  1.0276],
        [ 0.3105,  0.7328,  0.2716,  ...,  0.1117,  0.4277,  0.8033]])

In [69]:
X_train_tensor.shape

torch.Size([455, 30])

In [70]:
X_test_tensor.shape

torch.Size([114, 30])

Defining the model - NN

In [ ]:
class NN(nn.Module):
    
    def __init__(self, num_features): # X is input data
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )
        
        
    def forward(self, X):
        out = self.network(X)
        return out


Important params

In [82]:
lr = 0.1
epochs = 100

In [87]:
loss = nn.BCELoss()

Training Pipeline

In [98]:
# 1. Create model
model = NN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=lr)

## These steps in loop - epochs ##

for epoch in range(epochs):    

    # 2. Forward pass
    y_pred = model(X_train_tensor)
    # print(y_pred)
    
    # 3. Loss
    loss_res = loss(y_pred, y_train_tensor.unsqueeze(1))
    
    # 3.1 zero
    optimizer.zero_grad()
    
    # 4. Back pass
    loss_res.backward()
    
    # 5. Params update
    optimizer.step()
    optimizer.zero_grad()
    
    print(f"Epoch = {epoch + 1}, loss = {loss_res}")


Epoch = 1, loss = 0.6638050675392151
Epoch = 2, loss = 0.6391733884811401
Epoch = 3, loss = 0.6198517084121704
Epoch = 4, loss = 0.6035704016685486
Epoch = 5, loss = 0.5895398259162903
Epoch = 6, loss = 0.5769643187522888
Epoch = 7, loss = 0.5653984546661377
Epoch = 8, loss = 0.5546343922615051
Epoch = 9, loss = 0.5443690419197083
Epoch = 10, loss = 0.5346152782440186
Epoch = 11, loss = 0.5252442359924316
Epoch = 12, loss = 0.5161618590354919
Epoch = 13, loss = 0.5074969530105591
Epoch = 14, loss = 0.49914273619651794
Epoch = 15, loss = 0.4910062253475189
Epoch = 16, loss = 0.48308366537094116
Epoch = 17, loss = 0.4754335582256317
Epoch = 18, loss = 0.4679746627807617
Epoch = 19, loss = 0.46072468161582947
Epoch = 20, loss = 0.4537017047405243
Epoch = 21, loss = 0.4468183219432831
Epoch = 22, loss = 0.4400998055934906
Epoch = 23, loss = 0.4335617423057556
Epoch = 24, loss = 0.42720410227775574
Epoch = 25, loss = 0.42103901505470276
Epoch = 26, loss = 0.4150295853614807
Epoch = 27, loss

Evaluation

In [99]:
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    acc = (y_pred == y_test_tensor).float().mean()
    
print(acc)


tensor(0.5194)
